# WaterGAP 2.2d — Diffuse Groundwater Recharge

Visualize the CF-corrected WaterGAP 2.2d `qrdif` (diffuse groundwater recharge)
dataset fetched from PANGAEA (doi:10.1594/PANGAEA.918447).

- **Variable:** `qrdif` — diffuse groundwater recharge (kg m⁻² s⁻¹)
- **Resolution:** 0.5° global grid
- **Period:** 1901–2016 (monthly)
- **CF fix-ups applied:** datetime64 time coordinate, WGS84 grid mapping, CF-1.6 conventions

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
NC_PATH = DATASTORE / "watergap22d" / "watergap22d_qrdif_cf.nc"


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"Grid: {ds.lat.size} lat x {ds.lon.size} lon")
print(f"Conventions: {ds.attrs.get('Conventions', 'N/A')}")
print(f"Grid mapping: {ds['qrdif'].attrs.get('grid_mapping', 'N/A')}")


## Single-month recharge map (global)

In [ ]:
TARGET_TIME = "2000-01-01"

da = ds["qrdif"].sel(time=TARGET_TIME, method="nearest")
actual_time = str(da.time.values)[:10]

fig, ax = plt.subplots(figsize=(16, 6))
da.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "kg m⁻² s⁻¹"})
ax.set_title(f"Diffuse Groundwater Recharge (qrdif) — {actual_time}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Stats for {actual_time}:")
print(f"  min:  {float(da.min()):.2e}")
print(f"  max:  {float(da.max()):.2e}")
print(f"  mean: {float(da.mean()):.2e}")
print(f"  NaN%: {float(da.isnull().mean()) * 100:.1f}%")


## CONUS subset — recharge for calibration period (2000–2009)

In [ ]:
# CONUS bounding box (matches fabric bbox_buffered)
conus = ds["qrdif"].sel(
    lat=slice(50.1, 23.9),
    lon=slice(-125.1, -65.9),
    time=slice("2000-01-01", "2009-12-31"),
)

mean_recharge = conus.mean(dim="time")

fig, ax = plt.subplots(figsize=(14, 8))
mean_recharge.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "kg m⁻² s⁻¹"})
ax.set_title("Mean Diffuse Groundwater Recharge — CONUS 2000–2009")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"CONUS 2000-2009 mean stats:")
print(f"  min:  {float(mean_recharge.min()):.2e}")
print(f"  max:  {float(mean_recharge.max()):.2e}")
print(f"  mean: {float(mean_recharge.mean()):.2e}")


## Seasonal comparison (CONUS, 2000–2009)

In [ ]:
seasons = {
    "DJF (Winter)": conus.sel(time=conus.time.dt.month.isin([12, 1, 2])),
    "MAM (Spring)": conus.sel(time=conus.time.dt.month.isin([3, 4, 5])),
    "JJA (Summer)": conus.sel(time=conus.time.dt.month.isin([6, 7, 8])),
    "SON (Fall)": conus.sel(time=conus.time.dt.month.isin([9, 10, 11])),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label, seasonal_data) in zip(axes.flatten(), seasons.items()):
    seasonal_mean = seasonal_data.mean(dim="time")
    seasonal_mean.plot(ax=ax, cmap="YlGnBu", robust=True, cbar_kwargs={"label": "kg m⁻² s⁻¹"})
    ax.set_title(f"{label} mean recharge")
    ax.set_aspect("equal")

fig.suptitle("Seasonal Mean Recharge — CONUS 2000–2009", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## Monthly time series (CONUS spatial mean)

In [ ]:
# Full period CONUS spatial mean
conus_full = ds["qrdif"].sel(
    lat=slice(50.1, 23.9),
    lon=slice(-125.1, -65.9),
)
monthly_mean = conus_full.mean(dim=["lat", "lon"])

fig, ax = plt.subplots(figsize=(16, 4))
monthly_mean.plot(ax=ax, color="#2980b9", linewidth=0.5)
ax.set_ylabel("Mean Recharge (kg m⁻² s⁻¹)")
ax.set_title("CONUS-mean monthly groundwater recharge — 1901–2016")
ax.axvspan("2000-01-01", "2009-12-31", alpha=0.15, color="orange", label="Calibration period")
ax.legend()
plt.tight_layout()
plt.show()


## Annual mean climatology (CONUS, 2000–2009)

In [ ]:
# Monthly climatology for the calibration period
monthly_clim = conus.groupby("time.month").mean(dim=["time", "lat", "lon"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_clim.month, monthly_clim.values, color="#2980b9", edgecolor="white")
ax.set_xlabel("Month")
ax.set_ylabel("Mean Recharge (kg m⁻² s⁻¹)")
ax.set_title("Monthly Climatology — CONUS 2000–2009")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
plt.tight_layout()
plt.show()


## CF compliance verification

In [ ]:
print("CF Compliance Checks:")
print(f"  Conventions attr:  {ds.attrs.get('Conventions', 'MISSING')}")
print(f"  Time dtype:        {ds.time.dtype}")
print(f"  Time first:        {ds.time.values[0]}")
print(f"  Time last:         {ds.time.values[-1]}")
print(f"  CRS variable:      {'crs' in ds.data_vars}")
if "crs" in ds.data_vars:
    print(f"  Grid mapping name: {ds['crs'].attrs.get('grid_mapping_name', 'MISSING')}")
    print(f"  CRS WKT:           {ds['crs'].attrs.get('crs_wkt', 'MISSING')[:60]}...")
print(f"  qrdif grid_mapping: {ds['qrdif'].attrs.get('grid_mapping', 'MISSING')}")
print(f"  qrdif units:       {ds['qrdif'].attrs.get('units', 'MISSING')}")
print(f"  qrdif long_name:   {ds['qrdif'].attrs.get('long_name', 'MISSING')}")


## Manifest provenance check

In [ ]:
import json

manifest_path = PROJECT / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    entry = manifest["sources"].get("watergap22d", {})
    print(f"Source key:   {entry.get('source_key', 'N/A')}")
    print(f"DOI:          {entry.get('doi', 'N/A')}")
    print(f"License:      {entry.get('license', 'N/A')}")
    print(f"Period:       {entry.get('period', 'N/A')}")
    print(f"Variables:    {entry.get('variables', 'N/A')}")
    print(f"File path:    {entry.get('file', {}).get('path', 'N/A')}")
    print(f"Raw path:     {entry.get('file', {}).get('raw_path', 'N/A')}")
    print(f"Size (bytes): {entry.get('file', {}).get('size_bytes', 'N/A')}")
    print(f"Downloaded:   {entry.get('file', {}).get('downloaded_utc', 'N/A')}")
else:
    print(f"manifest.json not found at {manifest_path}")


In [ ]:
ds.close()
